# 01 - Autograd
Goal: how PyTorch computes gradients automatically - `requires_grad`, the computation graph,
`.backward()` / `.grad` - and why every training loop needs `zero_grad()`.

- Autograd solves: finding, for every weight, which direction to nudge it to reduce loss.
- How: `requires_grad=True` makes PyTorch record every op (computation graph);
  `.backward()` walks it in reverse (chain rule).
- Gradients land in each tensor's `.grad`.

In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # y = x²
y.backward()        # compute dy/dx
print(x.grad)       # tensor(6.)- because dy/dx = 2x = 2·3


tensor(6.)


Autograd computes weights to reduce loss.
```requires_grad=True``` makes Pytorch record every operation to build a computation graph which ```.backward()``` walks in reverse via chain rule to deposit each gradient in tensor's ```.grad``` attribute. 

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x ** 2 + 4 *x + 1   # 6x + 4 
y.backward()
print(x.grad)

tensor(16.)


Because ```2.0``` is 6x+4 = 12 + 4 = 16

In [4]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(5.0)              # no requires_grad - data doesn't need gradients
y = w * x + b                      # a one-weight "model"
y.backward()
print(w.grad, b.grad)              # ∂y/∂w = x = 5, ∂y/∂b = 1

tensor(5.) tensor(1.)


The third one is a neural network in miniature: gradients flow to the parameters (w, b), not the data. In your real model, ```requires_grad=True``` is set automatically on all weights by ```nn.Module```.

In [5]:
x = torch.tensor(3.0, requires_grad=True)
(x ** 2).backward()
print(x.grad)        # 6
(x ** 2).backward()
print(x.grad)        # 12?! — it ADDED the new 6 to the old 6

tensor(6.)
tensor(12.)


```.backward()``` adds to .grad, never replaces. That's why every training loop contains ```optimizer.zero_grad()```. Need to wipe the slate before each batch otherwise model trains badly hence zero_grad.

#### Turning recording off:

In [6]:
with torch.no_grad():
    y = x * 2        # nothing recorded: faster, less memory

This is used during evaluation/inference when no training is done and graph is waste. 

## Exercises

### Ex 1

In [19]:
x = torch.tensor(5.0, requires_grad=True)
f = (x-2) ** 2
f.backward()
print(x.grad)



tensor(6.)


### Ex 2
## The accumulation trap
`.backward()` **adds** to `.grad`, never replaces → why training loops call `optimizer.zero_grad()`.
Forgetting it: old batches contaminate new gradients, silently.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
(x ** 2).backward()
print(x.grad)        # 6
(x ** 2).backward()
print(x.grad)        # 12 - accumulated!

tensor(6.)
tensor(12.)


### Ex 3 - which way to move
x.grad is positive → increasing x increases f → to shrink f, move x DOWN (toward the minimum at 2).
Gradient descent = every parameter takes a small step OPPOSITE its gradient.

### Ex 4 - mini gradient descent (a training loop in miniature)
- `loss.backward()` → gradient 2(x−3) lands in x.grad
- update wrapped in `no_grad`: weight updates are bookkeeping, not math to differentiate
- `x.grad.zero_()` =  wipes the gradient - backward() accumulates, so wipe after each step
- x walks 10 → 3 (the minimum). In real loops: update = `optimizer.step()`, wipe = `optimizer.zero_grad()`.

In [ ]:
x = torch.tensor(10.0, requires_grad=True)   # our one "weight", starts far from ideal

for step in range(20):
    loss = (x - 3) ** 2       # how wrong we are; smallest (0) at x=3
    loss.backward()           # compute d(loss)/dx → lands in x.grad; here 2(x−3)
    with torch.no_grad():     # pause recording: the update is bookkeeping, not math to differentiate
        x -= 0.1 * x.grad     # step OPPOSITE the gradient; 0.1 = learning rate
    x.grad.zero_()            # wipe the gradient, because .backward() ADDS (accumulation trap)

print(x)                      # ≈ 3 — it walked downhill from 10 to the minimum